In [2]:
"""
================================================================================
Experiment 35 Ablation Study (Depth 3,4,5) – Automated Run (Fixed padding)
================================================================================
This script runs the ablation study for three depths (3,4,5) sequentially.
For each depth, it performs patient-level 3-fold cross-validation for:
- Feature extraction: MFCC only (40 coefficients)
- Method: CNN (2D) features + XGBoost classifier with class weights
- Minor classes (Healthy, URTI) merged into 'Other'.
- Per‑class metrics (precision, recall, f1, fpr, fnr) are saved per fold and aggregated.

All output directories are separated by depth (e.g., exp35_ablation_depth3, ...)
Checkpointing works independently for each depth.
"""

import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    balanced_accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, GlobalAveragePooling2D, Input
import xgboost as xgb
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
import json
import pickle
import time
from datetime import timedelta
from joblib import Parallel, delayed
from tqdm import tqdm
import gc

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# =============================================================================
# Fixed global configuration (same for all depths)
# =============================================================================
SOUND_FILES_PATH = "E:\\FFF\\respiratory-sound-database\\Respiratory_Sound_Database\\Respiratory_Sound_Database\\audio_and_txt_files\\"
DIAGNOSIS_CSV_PATH = "E:\\FFF\\respiratory-sound-database\\Respiratory_Sound_Database\\Respiratory_Sound_Database\\patient_diagnosis.csv"

# Processing options (same for all runs)
USE_PARALLEL = False
N_JOBS = 8 if USE_PARALLEL else 1
MAX_FILES = None                # Set to a number for testing (e.g., 200)
DTYPE = np.float32
CNN_BATCH_SIZE = 32

# MCDM weights (same for all)
WEIGHTS = {
    'macro_f1': 0.249,
    'macro_recall': 0.249,
    'weighted_f1': 0.157,
    'weighted_recall': 0.096,
    'accuracy': 0.096,
    'balanced_accuracy': 0.058,
    'inverted_fpr': 0.058,
    'fnr': 0.037
}

# =============================================================================
# Helper functions (reused for each depth)
# =============================================================================
def save_progress(stage, fold=None, progress_file=None):
    """Save progress to the given JSON file."""
    progress = {'stage': stage}
    if fold is not None:
        progress['last_completed_fold'] = fold
    with open(progress_file, 'w') as f:
        json.dump(progress, f)

def load_progress(progress_file):
    """Load progress from JSON file. Returns (stage, last_fold)."""
    if os.path.exists(progress_file):
        with open(progress_file, 'r') as f:
            progress = json.load(f)
        return progress.get('stage', 1), progress.get('last_completed_fold', 0)
    return 1, 0

def log_time(stage_name, elapsed_seconds, timing_file):
    """Log elapsed time to a file and print."""
    line = f"{stage_name}: {timedelta(seconds=elapsed_seconds)} (hh:mm:ss)\n"
    with open(timing_file, 'a') as f:
        f.write(line)
    print(line.strip())

def extract_features_single(file_path, max_pad_len=862):
    """Extract MFCC (40 coefficients) from a single audio file."""
    try:
        audio, sr = sf.read(file_path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        target_len = 20 * sr
        if len(audio) > target_len:
            audio = audio[:target_len]
        elif len(audio) < target_len:
            repeats = target_len // len(audio) + 1
            audio = np.tile(audio, repeats)[:target_len]

        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40).astype(DTYPE)

        if mfccs.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]

        return mfccs
    except Exception as e:
        print(f"\n!!! Error processing {file_path}: {e}")
        return None

def per_class_metrics(y_true, y_pred, class_names):
    """Compute per‑class precision, recall, f1, fpr, fnr."""
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    FP = cm.sum(axis=0) - np.diag(cm)
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    with np.errstate(divide='ignore', invalid='ignore'):
        FPR = np.divide(FP, FP + TN, where=(FP+TN) > 0)
        FNR = np.divide(FN, FN + TP, where=(FN+TP) > 0)
        FPR = np.nan_to_num(FPR, nan=0.0)
        FNR = np.nan_to_num(FNR, nan=0.0)

    result = {}
    for i, name in enumerate(class_names):
        result[name] = {
            'precision': precision[i],
            'recall': recall[i],
            'f1': f1[i],
            'fpr': FPR[i],
            'fnr': FNR[i]
        }
    return result

def build_cnn_depth(input_shape=(40, 862, 1), num_blocks=4):
    """
    Build a 2D CNN with the specified number of convolutional blocks.
    Each block: Conv2D(16*2**i, kernel_size=2, padding='same') + MaxPooling2D + Dropout(0.2)
    After blocks: GlobalAveragePooling2D()
    Using padding='same' prevents negative dimensions with deep architectures.
    """
    model = Sequential()
    model.add(Input(shape=input_shape))
    filters = 16
    for i in range(num_blocks):
        model.add(Conv2D(filters, kernel_size=2, activation='relu', padding='same'))
        model.add(MaxPooling2D(pool_size=2))
        model.add(Dropout(0.2))
        filters *= 2
    model.add(GlobalAveragePooling2D())
    return model

# =============================================================================
# Function that runs a single depth experiment
# =============================================================================
def run_experiment_for_depth(depth):
    print("\n" + "=" * 70)
    print(f"STARTING EXPERIMENT FOR DEPTH = {depth}")
    print("=" * 70)

    # Output directory specific to this depth
    OUTPUT_DIR = f"E:\\Results\\exp35_ablation_depth{depth}\\"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # File paths for this depth
    FEATURES_RAW_FILE = os.path.join(OUTPUT_DIR, "features_raw.npy")
    LABELS_RAW_FILE = os.path.join(OUTPUT_DIR, "labels_raw.npy")
    PATIENT_IDS_RAW_FILE = os.path.join(OUTPUT_DIR, "patient_ids_raw.npy")
    CLEANED_DATA_FILE = os.path.join(OUTPUT_DIR, "cleaned_data.pkl")
    PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")
    RESULTS_FILE = os.path.join(OUTPUT_DIR, "fold_results.csv")
    PER_CLASS_RESULTS_FILE = os.path.join(OUTPUT_DIR, "per_class_results.csv")
    TIMING_FILE = os.path.join(OUTPUT_DIR, "timing_log.txt")

    current_stage, last_fold = load_progress(PROGRESS_FILE)
    print(f"Depth {depth}: resuming from stage {current_stage}, last completed fold: {last_fold}")

    # -------------------------------------------------------------------------
    # STAGE 1: Extract raw features (identical for all depths)
    # -------------------------------------------------------------------------
    if current_stage == 1:
        stage_start = time.time()
        print("\n[Stage 1] Extracting MFCC features...")

        all_filenames = [f for f in os.listdir(SOUND_FILES_PATH)
                         if f.endswith('.wav') and os.path.isfile(os.path.join(SOUND_FILES_PATH, f))]
        if MAX_FILES is not None:
            filenames = all_filenames[:MAX_FILES]
            print(f"    Using first {len(filenames)} files (out of {len(all_filenames)} total).")
        else:
            filenames = all_filenames
            print(f"    Found {len(filenames)} audio files.")

        filepaths = [os.path.join(SOUND_FILES_PATH, f) for f in filenames]
        patient_ids = [int(name[:3]) for name in filenames]

        p_diag = pd.read_csv(DIAGNOSIS_CSV_PATH, header=None)
        diagnosis_dict = dict(zip(p_diag[0], p_diag[1]))
        labels = [diagnosis_dict.get(pid, None) for pid in patient_ids]

        if USE_PARALLEL:
            print(f"    Using {N_JOBS} parallel workers...")
            results = Parallel(n_jobs=N_JOBS, verbose=10)(
                delayed(extract_features_single)(fp) for fp in tqdm(filepaths, desc="Feature extraction")
            )
        else:
            print(f"    Using serial processing...")
            results = []
            for fp in tqdm(filepaths, desc="Feature extraction"):
                results.append(extract_features_single(fp))
                gc.collect()

        features = []
        valid_indices = []
        error_files = []
        for i, feat in enumerate(results):
            if feat is not None:
                features.append(feat)
                valid_indices.append(i)
            else:
                error_files.append(filepaths[i])

        if error_files:
            print(f"    Errors occurred on {len(error_files)} files. First few: {error_files[:5]}")
        print(f"    Extraction complete: {len(features)} successful, {len(error_files)} errors.")
        if len(features) == 0:
            raise RuntimeError("No features extracted.")

        features = np.array(features, dtype=DTYPE)  # (n_samples, 40, 862)
        patient_ids = [patient_ids[i] for i in valid_indices]
        labels = [labels[i] for i in valid_indices]

        np.save(FEATURES_RAW_FILE, features)
        np.save(LABELS_RAW_FILE, labels)
        np.save(PATIENT_IDS_RAW_FILE, patient_ids)
        print(f"    Raw features saved to {FEATURES_RAW_FILE}")

        elapsed = time.time() - stage_start
        log_time("Stage 1 - Feature extraction", elapsed, TIMING_FILE)
        save_progress(2, progress_file=PROGRESS_FILE)
        current_stage = 2  # manually update for the next blocks

    # -------------------------------------------------------------------------
    # STAGE 2: Clean and prepare data (same for all depths)
    # -------------------------------------------------------------------------
    if current_stage == 2:
        stage_start = time.time()
        print("\n[Stage 2] Cleaning data...")

        features = np.load(FEATURES_RAW_FILE, allow_pickle=True).astype(DTYPE)
        labels = np.load(LABELS_RAW_FILE, allow_pickle=True).tolist()
        patient_ids = np.load(PATIENT_IDS_RAW_FILE, allow_pickle=True).tolist()

        valid = [i for i, lbl in enumerate(labels) if lbl is not None]
        features = features[valid]
        patient_ids = [patient_ids[i] for i in valid]
        labels = [labels[i] for i in valid]

        new_features = []
        new_labels = []
        new_patient_ids = []
        for f, lbl, pid in zip(features, labels, patient_ids):
            if lbl in ['Asthma', 'LRTI']:
                continue
            elif lbl in ['Healthy', 'URTI']:
                new_labels.append('Other')
                new_features.append(f)
                new_patient_ids.append(pid)
            else:
                new_labels.append(lbl)
                new_features.append(f)
                new_patient_ids.append(pid)

        features = np.array(new_features, dtype=DTYPE)
        labels = new_labels
        patient_ids = new_patient_ids

        print(f"    Remaining samples after cleaning: {len(features)}")
        print("    New class distribution:")
        print(Counter(labels))

        # Add channel dimension for 2D CNN
        features = features[..., np.newaxis]  # (n_samples, 40, 862, 1)

        cleaned_data = {
            'features': features,
            'labels': labels,
            'patient_ids': patient_ids
        }
        with open(CLEANED_DATA_FILE, 'wb') as f:
            pickle.dump(cleaned_data, f)
        print(f"    Cleaned data saved to {CLEANED_DATA_FILE}")

        elapsed = time.time() - stage_start
        log_time("Stage 2 - Data cleaning", elapsed, TIMING_FILE)
        save_progress(3, progress_file=PROGRESS_FILE)
        current_stage = 3

    # -------------------------------------------------------------------------
    # STAGE 3: Cross-validation with variable-depth CNN + class weights
    # -------------------------------------------------------------------------
    if current_stage == 3:
        stage_start = time.time()
        print(f"\n[Stage 3] Running cross‑validation with depth {depth}...")

        with open(CLEANED_DATA_FILE, 'rb') as f:
            cleaned = pickle.load(f)
        features = cleaned['features'].astype(DTYPE)
        labels = cleaned['labels']
        patient_ids = cleaned['patient_ids']

        df = pd.DataFrame({
            'features': list(features),
            'label': labels,
            'patient_id': patient_ids
        })

        unique_patients = df.groupby('patient_id')['label'].first().reset_index()
        le = LabelEncoder()
        unique_patients['label_enc'] = le.fit_transform(unique_patients['label'])

        print(f"    Unique patients: {len(unique_patients)}")
        print(f"    Class distribution among patients:\n{unique_patients['label'].value_counts()}")

        n_splits = 3
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

        if os.path.exists(RESULTS_FILE):
            fold_results_df = pd.read_csv(RESULTS_FILE)
            fold_results = fold_results_df.to_dict('records')
            completed_folds = set(fold_results_df['fold'].tolist())
        else:
            fold_results = []
            completed_folds = set()

        if os.path.exists(PER_CLASS_RESULTS_FILE):
            per_class_df = pd.read_csv(PER_CLASS_RESULTS_FILE)
            per_class_fold_data = per_class_df.to_dict('records')
        else:
            per_class_fold_data = []

        for fold, (train_pat_idx, test_pat_idx) in enumerate(skf.split(
                unique_patients['patient_id'], unique_patients['label_enc'])):
            fold_num = fold + 1
            if fold_num in completed_folds:
                print(f"\n--- Fold {fold_num} already completed, skipping ---")
                continue

            fold_start = time.time()
            print(f"\n--- Fold {fold_num}/{n_splits} ---")

            train_patients = unique_patients.iloc[train_pat_idx]['patient_id'].values
            test_patients = unique_patients.iloc[test_pat_idx]['patient_id'].values

            train_df = df[df['patient_id'].isin(train_patients)]
            test_df = df[df['patient_id'].isin(test_patients)]

            X_train = np.stack(train_df['features'].values).astype(DTYPE)
            X_test = np.stack(test_df['features'].values).astype(DTYPE)
            y_train_str = train_df['label'].values
            y_test_str = test_df['label'].values

            le_fold = LabelEncoder()
            y_train = le_fold.fit_transform(y_train_str)
            y_test = le_fold.transform(y_test_str)
            classes = np.unique(y_train)

            print(f"    Train samples: {len(X_train)}, Test samples: {len(X_test)}")
            print(f"    Train patients: {len(train_patients)}, Test patients: {len(test_patients)}")
            print(f"    Classes in train: {le_fold.inverse_transform(classes)}")

            # --- Build CNN with variable depth (fixed padding) ---
            cnn = build_cnn_depth(input_shape=(40, 862, 1), num_blocks=depth)
            cnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
            cnn.summary()

            X_train_feat = cnn.predict(X_train, verbose=0, batch_size=CNN_BATCH_SIZE)
            X_test_feat = cnn.predict(X_test, verbose=0, batch_size=CNN_BATCH_SIZE)
            X_train_feat = X_train_feat.reshape(X_train_feat.shape[0], -1).astype(DTYPE)
            X_test_feat = X_test_feat.reshape(X_test_feat.shape[0], -1).astype(DTYPE)

            # --- Class weights ---
            class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
            sample_weight = np.array([class_weights[y] for y in y_train])

            # --- XGBoost training with sample weights ---
            xgb_model = xgb.XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.8,
                gamma=0.1,
                use_label_encoder=False,
                eval_metric='mlogloss',
                random_state=42
            )
            xgb_model.fit(X_train_feat, y_train, sample_weight=sample_weight)
            y_pred = xgb_model.predict(X_test_feat)

            # --- Metrics ---
            cm = confusion_matrix(y_test, y_pred, labels=classes)
            FP = cm.sum(axis=0) - np.diag(cm)
            FN = cm.sum(axis=1) - np.diag(cm)
            TP = np.diag(cm)
            TN = cm.sum() - (FP + FN + TP)

            with np.errstate(divide='ignore', invalid='ignore'):
                FPR_per_class = np.divide(FP, FP + TN, where=(FP+TN) > 0)
                FNR_per_class = np.divide(FN, FN + TP, where=(FN+TP) > 0)
                FPR_per_class = np.nan_to_num(FPR_per_class, nan=0.0)
                FNR_per_class = np.nan_to_num(FNR_per_class, nan=0.0)

            macro_fpr = np.mean(FPR_per_class)
            macro_fnr = np.mean(FNR_per_class)

            fold_metrics = {
                'fold': fold_num,
                'accuracy': accuracy_score(y_test, y_pred),
                'macro_f1': f1_score(y_test, y_pred, average='macro', zero_division=0),
                'macro_recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
                'weighted_f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
                'weighted_recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
                'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
                'fpr': macro_fpr,
                'fnr': macro_fnr
            }
            fold_results.append(fold_metrics)

            class_names = le_fold.classes_
            per_class = per_class_metrics(y_test, y_pred, class_names)
            for class_name, metrics in per_class.items():
                per_class_fold_data.append({
                    'fold': fold_num,
                    'class': class_name,
                    'precision': metrics['precision'],
                    'recall': metrics['recall'],
                    'f1': metrics['f1'],
                    'fpr': metrics['fpr'],
                    'fnr': metrics['fnr']
                })

            pd.DataFrame(fold_results).to_csv(RESULTS_FILE, index=False)
            pd.DataFrame(per_class_fold_data).to_csv(PER_CLASS_RESULTS_FILE, index=False)
            save_progress(3, fold_num, progress_file=PROGRESS_FILE)

            fold_elapsed = time.time() - fold_start
            print(f"    Fold {fold_num} completed in {timedelta(seconds=fold_elapsed)}.")

        elapsed = time.time() - stage_start
        log_time("Stage 3 - Cross-validation", elapsed, TIMING_FILE)
        print("\nAll folds completed.")
        save_progress(4, progress_file=PROGRESS_FILE)
        current_stage = 4

    # -------------------------------------------------------------------------
    # STAGE 4: Aggregate final results
    # -------------------------------------------------------------------------
    if current_stage == 4 or (current_stage >= 3 and os.path.exists(RESULTS_FILE)):
        print("\n[Stage 4] Aggregating final results...")
        fold_results_df = pd.read_csv(RESULTS_FILE)
        fold_results = fold_results_df.to_dict('records')

        avg_metrics = {}
        for metric in ['accuracy', 'macro_f1', 'macro_recall', 'weighted_f1',
                       'weighted_recall', 'balanced_accuracy', 'fpr', 'fnr']:
            values = [f[metric] for f in fold_results]
            avg_metrics[metric] = np.mean(values)
            avg_metrics[f'{metric}_std'] = np.std(values)

        inverted_fpr = 1 - avg_metrics['fpr']
        mcdm_score = (
            avg_metrics['macro_f1'] * WEIGHTS['macro_f1'] +
            avg_metrics['macro_recall'] * WEIGHTS['macro_recall'] +
            avg_metrics['weighted_f1'] * WEIGHTS['weighted_f1'] +
            avg_metrics['weighted_recall'] * WEIGHTS['weighted_recall'] +
            avg_metrics['accuracy'] * WEIGHTS['accuracy'] +
            avg_metrics['balanced_accuracy'] * WEIGHTS['balanced_accuracy'] +
            inverted_fpr * WEIGHTS['inverted_fpr'] +
            avg_metrics['fnr'] * WEIGHTS['fnr']
        )

        print("\n" + "=" * 50)
        print(f"FINAL AVERAGED METRICS (depth {depth})")
        print("=" * 50)
        for metric in ['accuracy', 'macro_f1', 'macro_recall', 'weighted_f1',
                       'weighted_recall', 'balanced_accuracy', 'fpr', 'fnr']:
            print(f"  {metric:18s}: {avg_metrics[metric]:.4f} ± {avg_metrics[f'{metric}_std']:.4f}")
        print(f"  inverted_fpr        : {inverted_fpr:.4f}")
        print("-" * 40)
        print(f"  *** MCDM SCORE      : {mcdm_score:.4f} ***")
        print("=" * 50)

        summary = {
            'metric': ['accuracy', 'macro_f1', 'macro_recall', 'weighted_f1',
                       'weighted_recall', 'balanced_accuracy', 'fpr', 'fnr', 'mcdm_score'],
            'mean': [avg_metrics['accuracy'], avg_metrics['macro_f1'], avg_metrics['macro_recall'],
                     avg_metrics['weighted_f1'], avg_metrics['weighted_recall'],
                     avg_metrics['balanced_accuracy'], avg_metrics['fpr'], avg_metrics['fnr'], mcdm_score],
            'std': [avg_metrics['accuracy_std'], avg_metrics['macro_f1_std'], avg_metrics['macro_recall_std'],
                    avg_metrics['weighted_f1_std'], avg_metrics['weighted_recall_std'],
                    avg_metrics['balanced_accuracy_std'], avg_metrics['fpr_std'], avg_metrics['fnr_std'], np.nan]
        }
        summary_df = pd.DataFrame(summary)
        summary_df.to_csv(os.path.join(OUTPUT_DIR, "final_summary.csv"), index=False)
        print(f"\nSummary saved to {os.path.join(OUTPUT_DIR, 'final_summary.csv')}")

        if os.path.exists(TIMING_FILE):
            print("\n" + "=" * 50)
            print("TIMING LOG")
            with open(TIMING_FILE, 'r') as f:
                print(f.read())

    print("\n" + "=" * 60)
    print(f"Experiment for depth {depth} completed.")
    print("=" * 60)

# =============================================================================
# Main loop: run depths 3, 4, 5 sequentially
# =============================================================================
if __name__ == "__main__":
    for depth in [3, 4, 5]:
        run_experiment_for_depth(depth)
        # Optional: small delay or clear session to free memory between runs
        tf.keras.backend.clear_session()
        gc.collect()


STARTING EXPERIMENT FOR DEPTH = 3
Depth 3: resuming from stage 4, last completed fold: 0

[Stage 4] Aggregating final results...

FINAL AVERAGED METRICS (depth 3)
  accuracy          : 0.8932 ± 0.0284
  macro_f1          : 0.5023 ± 0.0517
  macro_recall      : 0.5112 ± 0.0681
  weighted_f1       : 0.8910 ± 0.0263
  weighted_recall   : 0.8932 ± 0.0284
  balanced_accuracy : 0.5112 ± 0.0681
  fpr               : 0.0637 ± 0.0165
  fnr               : 0.4888 ± 0.0681
  inverted_fpr        : 0.9363
----------------------------------------
  *** MCDM SCORE      : 0.6658 ***

Summary saved to E:\Results\exp35_ablation_depth3\final_summary.csv

TIMING LOG
Stage 1 - Feature extraction: 0:03:30.412899 (hh:mm:ss)
Stage 2 - Data cleaning: 0:00:00.466790 (hh:mm:ss)
Stage 3 - Cross-validation: 0:00:07.413745 (hh:mm:ss)


Experiment for depth 3 completed.

STARTING EXPERIMENT FOR DEPTH = 4
Depth 4: resuming from stage 4, last completed fold: 0

[Stage 4] Aggregating final results...

FINAL AVERAGED M

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 40, 862, 16)    │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 20, 431, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 431, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 20, 431, 32)    │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 215, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 215, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 10, 215, 64)    │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 5, 107, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 5, 107, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 5, 107, 128)    │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 2, 53, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 2, 53, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 2, 53, 256)     │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 1, 26, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1, 26, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 174,640 (682.19 KB)

 Trainable params: 174,640 (682.19 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_10 (Conv2D)              │ (None, 40, 862, 16)    │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 20, 431, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 20, 431, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 20, 431, 32)    │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 10, 215, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 10, 215, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 10, 215, 64)    │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 5, 107, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 5, 107, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 5, 107, 128)    │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 2, 53, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 2, 53, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 2, 53, 256)     │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 1, 26, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 1, 26, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 174,640 (682.19 KB)

 Trainable params: 174,640 (682.19 KB)

 Non-trainable params: 0 (0.00 B)

    Fold 3 completed in 0:00:03.737221.
Stage 3 - Cross-validation: 0:00:10.752978 (hh:mm:ss)

All folds completed.

[Stage 4] Aggregating final results...

FINAL AVERAGED METRICS (depth 5)
  accuracy          : 0.8948 ± 0.0255
  macro_f1          : 0.4871 ± 0.1098
  macro_recall      : 0.5135 ± 0.1167
  weighted_f1       : 0.8882 ± 0.0269
  weighted_recall   : 0.8948 ± 0.0255
  balanced_accuracy : 0.5135 ± 0.1167
  fpr               : 0.0697 ± 0.0218
  fnr               : 0.4865 ± 0.1167
  inverted_fpr        : 0.9303
----------------------------------------
  *** MCDM SCORE      : 0.6621 ***

Summary saved to E:\Results\exp35_ablation_depth5\final_summary.csv

TIMING LOG
Stage 1 - Feature extraction: 0:03:33.754011 (hh:mm:ss)
Stage 2 - Data cleaning: 0:00:00.351834 (hh:mm:ss)
Stage 3 - Cross-validation: 0:00:10.752978 (hh:mm:ss)


Experiment for depth 5 completed.
